# EDA: Deteccion de Leucemia a partir de Imagenes de Celulas Sanguineas (v2)

Version adaptada a los datasets **originales publicados por los autores**:

- **WBCAtt / PBC dataset** (celulas sanas, `LY_3945.jpg`, etc.) -> se distribuye como 3 CSV de atributos (`pbc_attr_v1_train.csv`, `_val.csv`, `_test.csv`) separados de las imagenes.
  Repo: https://github.com/apple2373/wbcatt
- **LeukemiaAttri** (celulas con leucemia, `1_16_111_400_ALL.png`, etc.) -> cada carpeta `Grupo_Magnificacion_Control` (ej. `H_10X_C1`) contiene `Images/`, `json_labels/` y `txt_labels/`.
  Repo: https://github.com/intelligentMachines-ITU/Blood-Cancer-Dataset-Lukemia-Attri-MICCAI-2024

**Que cambia respecto a la version anterior del EDA:**
1. Ya no se recorre `HEALTHY_DIR` como carpetas-por-clase; se leen los CSV de WBCAtt y se resuelve la ruta de cada imagen.
2. En LeukemiaAttri la subcarpeta de imagenes paso de `images/train` a `Images`.
3. Se agregan secciones nuevas de EDA de **atributos morfologicos**, que antes no existian porque no teniamos esas anotaciones.

> Las rutas de la seccion 1 son las unicas que casi seguro debes ajustar a tu maquina. Todo lo demas deberia funcionar sin tocarlo, pero cada bloque imprime lo que encuentra para que puedas detectar rapido si algo no calza con tu estructura real.


## 1. Configuracion de entorno y rutas

In [ ]:
# %pip install pandas numpy matplotlib pillow scikit-image statsmodels opencv-python
from pathlib import Path
from collections import Counter, defaultdict
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import time

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

def progreso(i, total, inicio, cada=2000, etiqueta=""):
    """Imprime avance cada 'cada' iteraciones, para que los bucles largos
    (miles de imagenes sobre OneDrive) no parezcan congelados."""
    if total and (i % cada == 0 or i == total):
        transcurrido = time.time() - inicio
        print(f"  [{etiqueta}] {i}/{total} ({i / total * 100:.1f}%) - {transcurrido:.0f}s transcurridos")

# ------------------------------------------------------------
# WBCAtt (celulas sanas): carpeta con los 3 CSV de atributos
# ------------------------------------------------------------
WBCATT_DIR = Path(r"C:\Users\karin\OneDrive - Instituto Politecnico Nacional\Desktop\TT\Dataset Andres\WBCAtt")

WBCATT_CSVS = {
    "train": WBCATT_DIR / "pbc_attr_v1_train.csv",
    "val":   WBCATT_DIR / "pbc_attr_v1_val.csv",
    "test":  WBCATT_DIR / "pbc_attr_v1_test.csv",
}

# Carpeta donde viven FISICAMENTE las imagenes del dataset PBC
# (el CSV trae una columna "path" relativa a esta carpeta).
# Si no la tienes descargada por separado, ajusta esta ruta cuando
# la tengas; el resto del notebook sigue funcionando con el CSV solo,
# pero no podra abrir/analizar pixeles hasta que esto apunte a las imagenes reales.
WBCATT_IMAGES_DIR = Path(r"C:\Users\karin\OneDrive - Instituto Politecnico Nacional\Desktop\TT\Dataset Andres\WBCAtt\PBC_dataset_normal_DIB")

# ------------------------------------------------------------
# LeukemiaAttri (celulas con leucemia): carpetas Grupo_Magnificacion_Control
# ------------------------------------------------------------
# IMPORTANTE: cuando descargas una carpeta compartida grande desde OneDrive/Drive,
# suele venir partida en varias sub-descargas (ej. "..._1-001", "..._1-002", ...),
# cada una con su propia copia del arbol "LeukemiaAttri_Dataset/H_10X_C1/Images/...".
# Por eso NO apuntamos a una parte especifica: apuntamos a la carpeta que las
# contiene a TODAS ("LeukemiaAttri") y mas abajo buscamos de forma recursiva
# cualquier carpeta con patron Grupo_Magnificacion_Control (H_10X_C1, L_40X_C2, etc.),
# sin importar en cual de las partes este.
LEUKEMIA_ROOT = Path(r"C:\Users\karin\OneDrive - Instituto Politecnico Nacional\Desktop\TT\Dataset Andres\LeukemiaAttri")

print("WBCATT_DIR       :", WBCATT_DIR, "->", WBCATT_DIR.exists())
for split, p in WBCATT_CSVS.items():
    print(f"  CSV {split:<5}:", p, "->", p.exists())
print("WBCATT_IMAGES_DIR:", WBCATT_IMAGES_DIR, "->", WBCATT_IMAGES_DIR.exists())

print("\nLEUKEMIA_ROOT     :", LEUKEMIA_ROOT, "->", LEUKEMIA_ROOT.exists())
if LEUKEMIA_ROOT.exists():
    partes = [f.name for f in sorted(LEUKEMIA_ROOT.iterdir()) if f.is_dir()]
    print(f"  Partes/subcarpetas de primer nivel encontradas ({len(partes)}):")
    for p in partes:
        print("   -", p)


## 2. Carga de WBCAtt (atributos + resolucion de rutas de imagen)

Los CSV traen (segun la documentacion oficial del dataset):

- `img_name`: nombre del archivo (identificador unico).
- `label`: una de las 5 clases WBC del dataset PBC (neutrophils, eosinophils, basophils, monocytes, lymphocytes).
- `path`: ruta de la imagen dentro del dataset PBC original.
- 11 columnas de atributos morfologicos: `cell_size`, `cell_shape`, `nucleus_shape`, `nuclear_cytoplasmic_ratio`, `chromatin_density`, `cytoplasm_vacuole`, `cytoplasm_texture`, `cytoplasm_colour`, `granule_type`, `granule_colour`, `granularity`.

Primero cargamos e inspeccionamos, porque el nombre exacto de columnas puede variar ligeramente segun la version del CSV que te compartieron.

In [ ]:
# Cargar los 3 splits y unirlos en un solo DataFrame con una columna "Split"
wbcatt_frames = []
for split, path in WBCATT_CSVS.items():
    if not path.exists():
        print(f"[AVISO] No se encontro {path}, se omite el split '{split}'.")
        continue
    df_split = pd.read_csv(path)
    df_split["Split"] = split
    wbcatt_frames.append(df_split)

if wbcatt_frames:
    wbcatt_df = pd.concat(wbcatt_frames, ignore_index=True)
else:
    wbcatt_df = pd.DataFrame()

print(f"Total de filas cargadas (WBCAtt): {len(wbcatt_df)}")
print("\nColumnas encontradas:")
print(list(wbcatt_df.columns))
print("\nPrimeras filas:")
wbcatt_df.head()


In [ ]:
# Columnas de atributos morfologicos esperadas (segun documentacion oficial WBCAtt)
ATTRIBUTE_COLS_WBCATT = [
    "cell_size", "cell_shape", "nucleus_shape", "nuclear_cytoplasmic_ratio",
    "chromatin_density", "cytoplasm_vacuole", "cytoplasm_texture",
    "cytoplasm_colour", "granule_type", "granule_colour", "granularity",
]

# Verificamos cuales de esas columnas realmente existen en tu CSV
# (por si el compañero exporto con nombres ligeramente distintos, ej. mayusculas)
cols_lower = {c.lower(): c for c in wbcatt_df.columns}
ATTRIBUTE_COLS_WBCATT = [cols_lower[c] for c in ATTRIBUTE_COLS_WBCATT if c in cols_lower]

LABEL_COL_WBCATT = cols_lower.get("label", None)
IMGNAME_COL_WBCATT = cols_lower.get("img_name", None)
PATH_COL_WBCATT = cols_lower.get("path", None)

print("Columnas de atributos detectadas:", ATTRIBUTE_COLS_WBCATT)
print("Columna de clase (label)        :", LABEL_COL_WBCATT)
print("Columna de nombre de archivo     :", IMGNAME_COL_WBCATT)
print("Columna de ruta                  :", PATH_COL_WBCATT)


In [ ]:
# Construir la ruta absoluta de cada imagen a partir de WBCATT_IMAGES_DIR + columna "path"
def resolver_ruta_wbcatt(fila):
    if PATH_COL_WBCATT and pd.notna(fila.get(PATH_COL_WBCATT)):
        return WBCATT_IMAGES_DIR / str(fila[PATH_COL_WBCATT])
    if IMGNAME_COL_WBCATT and pd.notna(fila.get(IMGNAME_COL_WBCATT)):
        return WBCATT_IMAGES_DIR / str(fila[IMGNAME_COL_WBCATT])
    return None

if not wbcatt_df.empty:
    wbcatt_df["Ruta_Absoluta"] = wbcatt_df.apply(resolver_ruta_wbcatt, axis=1)
    wbcatt_df["Existe"] = wbcatt_df["Ruta_Absoluta"].apply(lambda p: p.exists() if p is not None else False)

    print(f"Imagenes cuya ruta SI se encontro en disco : {wbcatt_df['Existe'].sum()} / {len(wbcatt_df)}")
    print(f"Imagenes cuya ruta NO se encontro en disco : {(~wbcatt_df['Existe']).sum()} / {len(wbcatt_df)}")

    if wbcatt_df["Existe"].sum() == 0:
        print("\n[AVISO] Ninguna ruta resolvio a un archivo real.")
        print("Revisa WBCATT_IMAGES_DIR y/o el contenido de la columna 'path' con:")
        print("  wbcatt_df[[PATH_COL_WBCATT, 'Ruta_Absoluta']].head(10)")


## 3. Carga de LeukemiaAttri (imagenes + labels)

Buscamos de forma **recursiva** dentro de `LEUKEMIA_ROOT` cualquier carpeta cuyo
nombre siga el patron `Grupo_Magnificacion_Control` (ej. `H_10X_C1`, `L_40X_C2`),
sin importar en cual de las partes descargadas (`..._1-001`, `..._1-002`, etc.)
este ubicada. Esto es lo que estaba fallando antes: apuntabamos a una sola parte
y las demas quedaban fuera, o el nivel de anidamiento no coincidia exactamente.

Dentro de cada carpeta de condicion encontrada, tomamos las imagenes de `Images/`
(o `images/` en minuscula, por si acaso). Dentro de `Images/` a veces hay
subcarpetas `train/` y `test/` con las imagenes adentro (en vez de estar
sueltas directamente en `Images/`); usamos busqueda recursiva para que
funcione sin importar cual de los dos casos aplique, y guardamos en que
subcarpeta (`Split`) estaba cada imagen por si te sirve luego.

In [ ]:
import re

PATRON_CARPETA_CONDICION = re.compile(r"^[HL]_\d+X_C\d+$")

def encontrar_carpetas_condicion(root):
    """Busca recursivamente, en cualquier nivel bajo 'root', carpetas cuyo nombre
    matchee Grupo_Magnificacion_Control (ej. H_10X_C1). Asi no importa si el
    dataset esta partido en varias sub-descargas de OneDrive/Drive."""
    encontradas = []
    if not root.exists():
        return encontradas
    for p in root.rglob("*"):
        if p.is_dir() and PATRON_CARPETA_CONDICION.match(p.name):
            encontradas.append(p)
    return sorted(encontradas, key=lambda p: str(p))

carpetas_condicion = encontrar_carpetas_condicion(LEUKEMIA_ROOT)
print(f"Carpetas de condicion (Grupo_Magnificacion_Control) encontradas: {len(carpetas_condicion)}")
for c in carpetas_condicion:
    print("  -", c)

if not carpetas_condicion:
    print("\n[AVISO] No se encontro ninguna carpeta con el patron esperado (ej. H_10X_C1).")
    print("Revisa LEUKEMIA_ROOT y confirma que el nombre de esas carpetas sea exactamente asi")
    print("(mayusculas, guion bajo). Si el patron real es distinto, ajusta PATRON_CARPETA_CONDICION.")


In [ ]:
def parse_carpeta_leukemia(nombre_carpeta):
    partes = nombre_carpeta.split("_")
    if len(partes) == 3:
        grupo, magnificacion, control = partes
        return grupo, magnificacion, control
    return None, None, None

leukemia_image_records = []

for folder in carpetas_condicion:
    images_dir = folder / "Images"
    if not images_dir.exists():
        # Compatibilidad por si alguna carpeta usa minuscula
        images_dir = folder / "images"
    if not images_dir.exists():
        print(f"[AVISO] {folder} no tiene subcarpeta Images/ ni images/, se omite.")
        continue

    grupo, magnificacion, control = parse_carpeta_leukemia(folder.name)

    # Usamos rglob porque dentro de "Images" puede haber subcarpetas como
    # train/ y test/ (o ninguna, segun la carpeta de condicion); asi funciona
    # sin importar el nivel de anidamiento.
    for image_path in images_dir.rglob("*"):
        if not image_path.is_file() or image_path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue
        leukemia_image_records.append({
            "Carpeta": folder.name,
            "Grupo": grupo,
            "Magnificacion": magnificacion,
            "Control": control,
            "Split": image_path.parent.name.lower() if image_path.parent != images_dir else "n/a",
            "Archivo": image_path.name,
            "Ruta_Absoluta": image_path,
            # Guardamos la carpeta de condicion completa (no la ruta al label)
            # porque los labels pueden estar como carpeta "txt_labels/" YA
            # extraida, o como archivo "txt_labels.zip" sin extraer.
            "Carpeta_Condicion_Path": folder,
        })

leukemia_df = pd.DataFrame(leukemia_image_records)
print(f"Total de imagenes encontradas en LeukemiaAttri: {len(leukemia_df)}")
if not leukemia_df.empty:
    print("\nCarpetas detectadas y conteo de imagenes:")
    print(leukemia_df["Carpeta"].value_counts())
    print("\nConteo por Split (train/test) si aplica:")
    print(leukemia_df["Split"].value_counts())

    # Aviso si alguna de las 12 combinaciones esperadas (H/L x 10X/40X/100X x C1/C2)
    # termino con cero imagenes en TODAS las partes revisadas: puede ser una carpeta
    # que aun no termina de sincronizar en OneDrive, o que realmente falta.
    combinaciones_esperadas = {
        f"{g}_{m}_{c}"
        for g in ["H", "L"]
        for m in ["10X", "40X", "100X"]
        for c in ["C1", "C2"]
    }
    combinaciones_encontradas = set(leukemia_df["Carpeta"].unique())
    faltantes = combinaciones_esperadas - combinaciones_encontradas
    if faltantes:
        print("\n[AVISO] Estas combinaciones no aportaron ninguna imagen en ninguna parte revisada:")
        for f in sorted(faltantes):
            print("  -", f)
        print("Revisa en el explorador si esas carpetas siguen sincronizando en OneDrive.")


## 4. Verificacion de integridad de imagenes (ambos datasets)

Se reutiliza una unica funcion generica que recibe una lista de rutas, en vez de asumir una estructura de carpetas fija. Esto es lo que realmente cambiaba entre versiones del dataset, asi que separarlo evita tener que reescribir esta logica cada vez que cambie la estructura de carpetas.

In [ ]:
def analizar_calidad_imagenes(rutas, nombre_dataset=""):
    format_counts = Counter()
    size_counts = Counter()
    mode_counts = Counter()
    corrupted_images = []
    total_images = 0

    inicio = time.time()
    total_a_revisar = len(rutas)
    print(f"Iniciando revision de {total_a_revisar} imagenes ({nombre_dataset})...")

    for i, image_path in enumerate(rutas, start=1):
        progreso(i, total_a_revisar, inicio, cada=2000, etiqueta=nombre_dataset)

        if image_path is None or not Path(image_path).exists():
            continue
        total_images += 1
        try:
            with Image.open(image_path) as img:
                img.verify()
            with Image.open(image_path) as img:
                format_counts[img.format] += 1
                size_counts[img.size] += 1
                mode_counts[img.mode] += 1
        except Exception as e:
            corrupted_images.append({"archivo": str(image_path), "error": str(e)})

    print(f"Terminado en {time.time() - inicio:.0f}s.")
    return total_images, format_counts, size_counts, mode_counts, corrupted_images


def imprimir_reporte_calidad(nombre, resultados):
    total, formatos, tamanios, modos, corruptas = resultados
    print(f"========== REVISION DE IMAGENES: {nombre.upper()} ==========")
    print(f"Total revisadas: {total}\n")
    print("--- Formatos ---", *[f"{k}: {v}" for k, v in formatos.most_common()], sep="\n")
    print("\n--- Dimensiones (top 10) ---", *[f"{k}: {v}" for k, v in tamanios.most_common(10)], sep="\n")
    print("\n--- Modos ---", *[f"{k}: {v}" for k, v in modos.most_common()], sep="\n")
    print(f"\n--- Imagenes con problemas ---\nTotal: {len(corruptas)}\n")


rutas_wbcatt = wbcatt_df.loc[wbcatt_df.get("Existe", pd.Series(dtype=bool)) == True, "Ruta_Absoluta"].tolist() if not wbcatt_df.empty else []
rutas_leukemia = leukemia_df["Ruta_Absoluta"].tolist() if not leukemia_df.empty else []

reporte_wbcatt = analizar_calidad_imagenes(rutas_wbcatt, "WBCAtt")
reporte_leukemia = analizar_calidad_imagenes(rutas_leukemia, "LeukemiaAttri")

imprimir_reporte_calidad("WBCAtt (sanas)", reporte_wbcatt)
imprimir_reporte_calidad("LeukemiaAttri", reporte_leukemia)


## 5. Distribucion de clases

- **WBCAtt**: distribucion de la columna `label` (tipo de celula sana).
- **LeukemiaAttri**: distribucion por carpeta (Grupo x Magnificacion x Control), igual que en la version anterior del EDA, solo que ahora el conteo de imagenes viene de `leukemia_df` en vez de recorrer `images/train`.

In [ ]:
# Distribucion WBCAtt
if not wbcatt_df.empty and LABEL_COL_WBCATT:
    df_wbcatt_clase = (
        wbcatt_df[LABEL_COL_WBCATT]
        .value_counts()
        .rename_axis("Clase")
        .reset_index(name="Imagenes")
    )
    df_wbcatt_clase["Porcentaje (%)"] = (df_wbcatt_clase["Imagenes"] / df_wbcatt_clase["Imagenes"].sum() * 100).round(2)
    print("========== DISTRIBUCION WBCAtt (por label) ==========")
    print(df_wbcatt_clase.to_string(index=False))
else:
    df_wbcatt_clase = pd.DataFrame()
    print("No hay datos de WBCAtt cargados todavia.")

# Distribucion LeukemiaAttri
if not leukemia_df.empty:
    df_leukemia_dist = (
        leukemia_df.groupby(["Grupo", "Magnificacion", "Control"])
        .size()
        .reset_index(name="Imagenes")
    )
    df_leukemia_dist["Porcentaje (%)"] = (df_leukemia_dist["Imagenes"] / df_leukemia_dist["Imagenes"].sum() * 100).round(2)
    print("\n========== DISTRIBUCION LeukemiaAttri ==========")
    print(df_leukemia_dist.sort_values("Imagenes", ascending=False).to_string(index=False))
else:
    df_leukemia_dist = pd.DataFrame()
    print("No hay datos de LeukemiaAttri cargados todavia.")


In [ ]:
# Graficas de distribucion
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

if not df_wbcatt_clase.empty:
    ax1.bar(df_wbcatt_clase["Clase"], df_wbcatt_clase["Imagenes"], color="skyblue")
    ax1.set_title("Distribucion de Clases (WBCAtt)")
    ax1.set_xticks(range(len(df_wbcatt_clase["Clase"])))
    ax1.set_xticklabels(df_wbcatt_clase["Clase"], rotation=45, ha="right")
    ax1.set_ylabel("Numero de Imagenes")

if not df_leukemia_dist.empty:
    df_mag = df_leukemia_dist.pivot_table(index="Magnificacion", columns="Grupo", values="Imagenes", aggfunc="sum", fill_value=0)
    df_mag = df_mag.reindex([m for m in ["10X", "40X", "100X"] if m in df_mag.index])
    df_mag.plot(kind="bar", ax=ax2, color=["#ff9999", "#66b3ff"])
    ax2.set_title("LeukemiaAttri: H vs L por Magnificacion")
    ax2.set_xlabel("Magnificacion")
    ax2.set_ylabel("Numero de Imagenes")
    ax2.tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()


## 6. EDA de atributos morfologicos - WBCAtt (NUEVO)

Esto no existia en la version anterior del EDA porque antes no teniamos anotaciones de atributos, solo imagenes por carpeta. Ahora podemos ver, por cada uno de los 11 atributos, como se distribuyen sus valores posibles, y como se relacionan con la clase (`label`).

In [ ]:
if not wbcatt_df.empty and ATTRIBUTE_COLS_WBCATT:
    n_attrs = len(ATTRIBUTE_COLS_WBCATT)
    n_cols = 3
    n_rows = -(-n_attrs // n_cols)  # ceil

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 4 * n_rows))
    axes = axes.flat if n_attrs > 1 else [axes]

    for ax, attr in zip(axes, ATTRIBUTE_COLS_WBCATT):
        conteo = wbcatt_df[attr].value_counts()
        ax.bar(conteo.index.astype(str), conteo.values, color="mediumpurple")
        ax.set_title(attr)
        ax.tick_params(axis="x", rotation=45)

    for ax in list(axes)[n_attrs:]:
        ax.axis("off")

    plt.suptitle("Distribucion de atributos morfologicos - WBCAtt", fontsize=16)
    plt.tight_layout()
    plt.show()
else:
    print("No se encontraron columnas de atributos para graficar.")


In [ ]:
# Tabla cruzada: atributo vs clase (label), para detectar si algun atributo
# esta fuertemente asociado a una clase especifica de celula
if not wbcatt_df.empty and LABEL_COL_WBCATT and ATTRIBUTE_COLS_WBCATT:
    for attr in ATTRIBUTE_COLS_WBCATT:
        print(f"\n=== {attr} vs {LABEL_COL_WBCATT} ===")
        tabla = pd.crosstab(wbcatt_df[LABEL_COL_WBCATT], wbcatt_df[attr], normalize="index").round(3) * 100
        print(tabla.to_string())


## 7. EDA de atributos - LeukemiaAttri (NUEVO)

Los `txt_labels` estan en formato YOLO extendido: `cls x y w h a1 a2 a3 a4 a5 a6` (clase + bounding box normalizado + 6 codigos de atributos morfologicos), a diferencia del YOLO estandar (`cls x y w h`). Parseamos todos los `.txt` para construir una tabla de detecciones (una fila por celula detectada, no por imagen) y poder analizar la distribucion de clases y atributos.

En tu caso `txt_labels` llega como **`txt_labels.zip` sin extraer** en algunas carpetas de condicion. La funcion de abajo soporta ambos casos (carpeta ya extraida o zip) sin que tengas que descomprimir nada a mano; para el zip, lee el archivo directamente desde adentro usando `zipfile`.

> Si tu version del dataset trae mas o menos columnas de atributo por linea, ajusta `N_ATTR_LEUKEMIA` abajo; el bloque te avisa si detecta un numero distinto de columnas al esperado.

In [ ]:
import zipfile

_indice_zip_cache = {}

def _indice_zip_txt_labels(zip_path):
    """Indexa una sola vez el contenido de un txt_labels.zip: {stem_del_archivo: nombre_dentro_del_zip}"""
    if zip_path not in _indice_zip_cache:
        indice = {}
        with zipfile.ZipFile(zip_path) as zf:
            for nombre in zf.namelist():
                if nombre.lower().endswith(".txt"):
                    stem = Path(nombre).stem
                    indice[stem] = nombre
        _indice_zip_cache[zip_path] = indice
    return _indice_zip_cache[zip_path]


def leer_lineas_txt_label(carpeta_condicion, stem):
    """Devuelve las lineas del .txt de labels para la imagen 'stem', ya sea que
    'txt_labels' este como carpeta extraida o como 'txt_labels.zip' sin extraer.
    Devuelve None si no encuentra el label para esa imagen."""
    carpeta_txt = carpeta_condicion / "txt_labels"
    if carpeta_txt.is_dir():
        archivo = carpeta_txt / f"{stem}.txt"
        if archivo.exists():
            with open(archivo, "r") as f:
                return [l.strip() for l in f if l.strip()]
        return None

    zip_txt = carpeta_condicion / "txt_labels.zip"
    if zip_txt.exists():
        indice = _indice_zip_txt_labels(zip_txt)
        nombre_en_zip = indice.get(stem)
        if nombre_en_zip is None:
            return None
        with zipfile.ZipFile(zip_txt) as zf, zf.open(nombre_en_zip) as f:
            contenido = f.read().decode("utf-8", errors="ignore")
        return [l.strip() for l in contenido.splitlines() if l.strip()]

    return None


N_ATTR_LEUKEMIA = 6  # segun el formato documentado: cls x y w h a1..a6

deteccion_records = []
conteo_columnas = Counter()
sin_label = 0

if not leukemia_df.empty:
    for _, fila in leukemia_df.iterrows():
        stem = Path(fila["Archivo"]).stem
        lineas = leer_lineas_txt_label(fila["Carpeta_Condicion_Path"], stem)

        if lineas is None:
            sin_label += 1
            continue

        for linea in lineas:
            valores = linea.split()
            conteo_columnas[len(valores)] += 1

            if len(valores) < 5:
                continue

            cls = valores[0]
            x, y, w, h = map(float, valores[1:5])
            atributos = valores[5:5 + N_ATTR_LEUKEMIA]

            registro = {
                "Carpeta": fila["Carpeta"],
                "Grupo": fila["Grupo"],
                "Magnificacion": fila["Magnificacion"],
                "Control": fila["Control"],
                "Archivo": fila["Archivo"],
                "Clase_YOLO": cls,
                "x": x, "y": y, "w": w, "h": h,
            }
            for i, val in enumerate(atributos, start=1):
                registro[f"a{i}"] = val

            deteccion_records.append(registro)

detecciones_df = pd.DataFrame(deteccion_records)

print("Cantidad de columnas por linea encontradas en los txt_labels (deberia concentrarse en 5 + N_ATTR_LEUKEMIA):")
print(conteo_columnas.most_common())
print(f"\nImagenes sin archivo de label encontrado: {sin_label}")
print(f"Total de detecciones (celulas) parseadas: {len(detecciones_df)}")
if not detecciones_df.empty:
    detecciones_df.head()


In [ ]:
# Distribucion de clases YOLO (subtipos de celula/blasto) y de cada atributo a1..a6
if not detecciones_df.empty:
    print("========== DISTRIBUCION DE Clase_YOLO ==========")
    print(detecciones_df["Clase_YOLO"].value_counts().to_string())

    attr_cols = [c for c in detecciones_df.columns if c.startswith("a") and c[1:].isdigit()]

    if attr_cols:
        n_cols = 3
        n_rows = -(-len(attr_cols) // n_cols)
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 4 * n_rows))
        axes = axes.flat if len(attr_cols) > 1 else [axes]

        for ax, col in zip(axes, attr_cols):
            conteo = detecciones_df[col].value_counts()
            ax.bar(conteo.index.astype(str), conteo.values, color="darkorange")
            ax.set_title(f"Distribucion de {col}")
            ax.tick_params(axis="x", rotation=45)

        for ax in list(axes)[len(attr_cols):]:
            ax.axis("off")

        plt.suptitle("Distribucion de atributos morfologicos - LeukemiaAttri", fontsize=16)
        plt.tight_layout()
        plt.show()
else:
    print("No hay detecciones parseadas todavia (revisa que N_ATTR_LEUKEMIA y las rutas txt_label_path sean correctas).")


## 8. Muestras de imagenes (ambos datasets)

In [ ]:
import random

def mostrar_muestras_aleatorias(rutas, titulo, num_muestras=4, seed=None):
    rutas = [r for r in rutas if r is not None and Path(r).exists()]
    if not rutas:
        print(f"No se encontraron imagenes para: {titulo}")
        return
    if seed is not None:
        random.seed(seed)
    muestras = random.sample(rutas, min(num_muestras, len(rutas)))

    fig, axes = plt.subplots(1, len(muestras), figsize=(15, 4))
    if len(muestras) == 1:
        axes = [axes]
    fig.suptitle(titulo, fontsize=16)

    for ax, img_path in zip(axes, muestras):
        img = Image.open(img_path)
        ax.imshow(img)
        ax.set_title(f"{img.size} | {img.mode}", fontsize=9)
        ax.axis("off")

    plt.tight_layout()
    plt.show()

mostrar_muestras_aleatorias(rutas_wbcatt, "Muestras aleatorias - WBCAtt (sanas)", num_muestras=4)
mostrar_muestras_aleatorias(rutas_leukemia, "Muestras aleatorias - LeukemiaAttri", num_muestras=4)


## 9. Deteccion de duplicados exactos (ambos datasets)

Igual que en la version anterior, pero ahora aplicado a las listas de rutas ya resueltas de cada dataset (`rutas_wbcatt`, `rutas_leukemia`), en vez de recorrer una estructura de carpetas fija.

In [ ]:
import hashlib

def calcular_hashes(rutas, nombre_dataset):
    registros = []
    inicio = time.time()
    total = len(rutas)
    for i, ruta in enumerate(rutas, start=1):
        progreso(i, total, inicio, cada=2000, etiqueta=f"hash-{nombre_dataset}")
        ruta = Path(ruta)
        if not ruta.exists():
            continue
        md5 = hashlib.md5()
        with open(ruta, "rb") as f:
            for chunk in iter(lambda: f.read(8192), b""):
                md5.update(chunk)
        registros.append({"Dataset": nombre_dataset, "Archivo": ruta.name, "Ruta": str(ruta), "Hash": md5.hexdigest()})
    return pd.DataFrame(registros)

hash_wbcatt_df = calcular_hashes(rutas_wbcatt, "WBCAtt")
hash_leukemia_df = calcular_hashes(rutas_leukemia, "LeukemiaAttri")
hash_df = pd.concat([hash_wbcatt_df, hash_leukemia_df], ignore_index=True)

if not hash_df.empty:
    duplicate_hashes = hash_df[hash_df["Hash"].duplicated(keep=False)].sort_values("Hash")

    print("========== DUPLICADOS EXACTOS ==========")
    print(f"Total de imagenes analizadas: {len(hash_df)}")
    print(f"Hashes unicos: {hash_df['Hash'].nunique()}")
    print(f"Imagenes involucradas en duplicados: {len(duplicate_hashes)}")

    if len(duplicate_hashes) > 0:
        print("\n--- Imagenes duplicadas ---")
        print(duplicate_hashes.to_string(index=False))
    else:
        print("\nNo se encontraron duplicados exactos.")
else:
    print("No hay imagenes para hashear todavia.")


## 10. Dimensiones, canales y color/brillo/contraste (ambos datasets)

Se generaliza la funcion de analisis de color para recibir una lista de `(ruta, dataset, grupo)` en vez de recorrer carpetas fijas, y asi funciona igual sin importar como cambie la estructura de carpetas en el futuro.

**Nota de rendimiento:** con ~26,000 imagenes de LeukemiaAttri viviendo en OneDrive, procesar el color/brillo de TODAS puede tardar varios minutos (o mas, si hay archivos "solo en la nube" que se descargan al vuelo). Por eso el bloque de color/brillo usa una muestra de `MUESTRA_COLOR` imagenes por grupo en vez del dataset completo; sube ese numero o ponlo en `None` cuando quieras el analisis exhaustivo, ya con tiempo de sobra. Todos los bucles largos de esta version imprimen su avance cada cierto numero de imagenes para que sepas que siguen corriendo.

In [ ]:
def construir_tabla_imagenes(rutas_con_metadata):
    """rutas_con_metadata: lista de dicts con al menos {'Ruta': ..., 'Dataset': ..., 'Grupo': ...}"""
    registros = []
    inicio = time.time()
    total = len(rutas_con_metadata)
    for i, item in enumerate(rutas_con_metadata, start=1):
        progreso(i, total, inicio, cada=2000, etiqueta="dimensiones")
        ruta = Path(item["Ruta"])
        if not ruta.exists():
            continue
        try:
            with Image.open(ruta) as image:
                width, height = image.size
                registros.append({
                    "Dataset": item.get("Dataset"),
                    "Grupo": item.get("Grupo"),
                    "Archivo": ruta.name,
                    "Ancho": width,
                    "Alto": height,
                    "Canales": len(image.getbands()),
                    "Formato": image.format,
                    "Aspect_Ratio": round(width / height, 3) if height else None,
                })
        except Exception as e:
            print(f"Error leyendo: {ruta} -> {e}")
    return pd.DataFrame(registros)


metadata_wbcatt = [
    {"Ruta": r, "Dataset": "WBCAtt", "Grupo": lbl}
    for r, lbl in zip(
        wbcatt_df.loc[wbcatt_df.get("Existe", pd.Series(dtype=bool)) == True, "Ruta_Absoluta"],
        wbcatt_df.loc[wbcatt_df.get("Existe", pd.Series(dtype=bool)) == True, LABEL_COL_WBCATT] if LABEL_COL_WBCATT else [],
    )
] if not wbcatt_df.empty else []

metadata_leukemia = [
    {"Ruta": r, "Dataset": "Leukemia", "Grupo": g}
    for r, g in zip(leukemia_df.get("Ruta_Absoluta", []), leukemia_df.get("Grupo", []))
] if not leukemia_df.empty else []

image_info_df = construir_tabla_imagenes(metadata_wbcatt + metadata_leukemia)
print(f"Total de imagenes en la tabla combinada: {len(image_info_df)}")
image_info_df.head()


In [ ]:
# COLOR, BRILLO Y CONTRASTE
def analyze_image_color(file_path):
    with Image.open(file_path) as img:
        img_rgb = img.convert("RGB")
        image_array = np.asarray(img_rgb, dtype=np.float32)

        mean_r = image_array[:, :, 0].mean()
        mean_g = image_array[:, :, 1].mean()
        mean_b = image_array[:, :, 2].mean()

        brightness = (
            0.299 * image_array[:, :, 0]
            + 0.587 * image_array[:, :, 1]
            + 0.114 * image_array[:, :, 2]
        )

        return {
            "Promedio_R": mean_r,
            "Promedio_G": mean_g,
            "Promedio_B": mean_b,
            "Brillo_Promedio": brightness.mean(),
            "Contraste_STD": brightness.std(),
            "Brillo_Minimo": brightness.min(),
            "Brillo_Maximo": brightness.max(),
        }


def construir_tabla_color(rutas_con_metadata):
    registros = []
    inicio = time.time()
    total = len(rutas_con_metadata)
    for i, item in enumerate(rutas_con_metadata, start=1):
        progreso(i, total, inicio, cada=2000, etiqueta="color/brillo")
        ruta = Path(item["Ruta"])
        if not ruta.exists():
            continue
        try:
            stats = analyze_image_color(ruta)
            stats.update({"Dataset": item.get("Dataset"), "Grupo": item.get("Grupo"), "Archivo": ruta.name})
            registros.append(stats)
        except Exception as e:
            print(f"Error leyendo: {ruta} -> {e}")
    return pd.DataFrame(registros)


# MUESTRA_COLOR: numero maximo de imagenes POR (Dataset, Grupo) a analizar.
# Con ~26,000 imagenes de LeukemiaAttri sobre OneDrive, procesar TODO puede
# tardar bastante. Deja un numero (ej. 500) para una primera pasada rapida,
# y ponlo en None cuando quieras correr el analisis completo (puede tardar
# varios minutos, sobre todo si hay que descargar archivos "solo en la nube").
MUESTRA_COLOR = 500

if MUESTRA_COLOR is None:
    rutas_para_color = metadata_wbcatt + metadata_leukemia
else:
    random.seed(42)
    rutas_para_color = []
    todas = metadata_wbcatt + metadata_leukemia
    grupos_presentes = {(item.get("Dataset"), item.get("Grupo")) for item in todas}
    for dataset, grupo in grupos_presentes:
        subset = [it for it in todas if it.get("Dataset") == dataset and it.get("Grupo") == grupo]
        rutas_para_color.extend(random.sample(subset, min(MUESTRA_COLOR, len(subset))))

print(f"Analizando color/brillo sobre {len(rutas_para_color)} imagenes (MUESTRA_COLOR={MUESTRA_COLOR}).")
color_df = construir_tabla_color(rutas_para_color)
print(f"Total de imagenes analizadas para color/brillo: {len(color_df)}")
color_df.head()


In [ ]:
# DISTRIBUCION DE BRILLO Y CONTRASTE POR DATASET/GRUPO
if not color_df.empty:
    color_df["Grupo_Display"] = color_df["Dataset"] + " - " + color_df["Grupo"].astype(str)
    grupos_unicos = color_df["Grupo_Display"].unique()

    plt.figure(figsize=(10, 6))
    for g in grupos_unicos:
        plt.hist(color_df.loc[color_df["Grupo_Display"] == g, "Brillo_Promedio"], bins=40, alpha=0.5, label=g)
    plt.title("Distribucion del brillo promedio")
    plt.xlabel("Brillo promedio")
    plt.ylabel("Numero de imagenes")
    plt.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(10, 6))
    for g in grupos_unicos:
        plt.hist(color_df.loc[color_df["Grupo_Display"] == g, "Contraste_STD"], bins=40, alpha=0.5, label=g)
    plt.title("Distribucion del contraste (STD del brillo)")
    plt.xlabel("Contraste")
    plt.ylabel("Numero de imagenes")
    plt.legend(fontsize=8)
    plt.tight_layout()
    plt.show()
else:
    print("color_df esta vacio, no hay nada que graficar todavia.")


## 11. Analisis de textura: entropia y GLCM (ambos datasets)

Se toma una muestra por grupo para no procesar el dataset completo (igual que en la version anterior), pero ahora usando `metadata_wbcatt` / `metadata_leukemia` como fuente de rutas.

In [ ]:
%pip install scikit-image

In [ ]:
from skimage.feature import graycomatrix, graycoprops
from skimage.measure import shannon_entropy

N_MUESTRA = 300
RANDOM_STATE = 42

def calcular_textura(rutas_con_metadata, n_muestra=N_MUESTRA, seed=RANDOM_STATE):
    random.seed(seed)
    muestra = random.sample(rutas_con_metadata, min(n_muestra, len(rutas_con_metadata)))

    registros = []
    inicio = time.time()
    total = len(muestra)
    for i, item in enumerate(muestra, start=1):
        progreso(i, total, inicio, cada=500, etiqueta="textura")
        ruta = Path(item["Ruta"])
        if not ruta.exists():
            continue
        try:
            with Image.open(ruta) as img:
                gris = np.array(img.convert("L"))

            entropia = shannon_entropy(gris)
            glcm = graycomatrix(gris, distances=[1], angles=[0], levels=256, symmetric=True, normed=True)

            registros.append({
                "Dataset": item.get("Dataset"),
                "Grupo": item.get("Grupo"),
                "Archivo": ruta.name,
                "Entropia": entropia,
                "GLCM_Contraste": graycoprops(glcm, "contrast")[0, 0],
                "GLCM_Homogeneidad": graycoprops(glcm, "homogeneity")[0, 0],
                "GLCM_Energia": graycoprops(glcm, "energy")[0, 0],
                "GLCM_Correlacion": graycoprops(glcm, "correlation")[0, 0],
            })
        except Exception as e:
            print(f"Error procesando textura de {ruta}: {e}")

    return pd.DataFrame(registros)

texture_df = calcular_textura(metadata_wbcatt + metadata_leukemia)
print(f"Total de imagenes analizadas para textura: {len(texture_df)}")
if not texture_df.empty:
    print(texture_df.groupby(["Dataset", "Grupo"]).size())


In [ ]:
# BOXPLOTS DE TEXTURA POR GRUPO
if not texture_df.empty:
    texture_df["Grupo_Display"] = texture_df["Dataset"] + " - " + texture_df["Grupo"].astype(str)
    grupos_orden = sorted(texture_df["Grupo_Display"].unique())

    metricas_textura = [
        ("Entropia", "Entropia"),
        ("GLCM_Contraste", "Contraste GLCM"),
        ("GLCM_Homogeneidad", "Homogeneidad GLCM"),
        ("GLCM_Energia", "Energia GLCM"),
        ("GLCM_Correlacion", "Correlacion GLCM"),
    ]

    for columna, titulo in metricas_textura:
        datos = [texture_df.loc[texture_df["Grupo_Display"] == g, columna].dropna().values for g in grupos_orden]

        plt.figure(figsize=(10, 6))
        plt.boxplot(datos, tick_labels=grupos_orden, showfliers=True)
        plt.title(f"Distribucion de {titulo} por grupo")
        plt.xticks(rotation=45, ha="right")
        plt.ylabel(titulo)
        plt.grid(axis="y", alpha=0.3)
        plt.tight_layout()
        plt.show()
else:
    print("texture_df esta vacio, no hay nada que graficar todavia.")


## 12. Pruebas estadisticas (Kruskal-Wallis + comparaciones por pares)

Igual que en la version anterior, pero ahora comparando entre todos los grupos presentes en `texture_df["Grupo_Display"]` en vez de asumir exactamente 3 grupos fijos (Healthy / Leukemia H / Leukemia L).

In [ ]:
%pip install statsmodels

In [ ]:
from scipy.stats import kruskal, mannwhitneyu
from statsmodels.stats.multitest import multipletests
from itertools import combinations

if not texture_df.empty:
    metricas_textura_cols = ["Entropia", "GLCM_Contraste", "GLCM_Homogeneidad", "GLCM_Energia", "GLCM_Correlacion"]
    grupos_orden = sorted(texture_df["Grupo_Display"].unique())

    print("=" * 78)
    print("     PRUEBA DE KRUSKAL-WALLIS (comparacion global entre todos los grupos)")
    print("=" * 78)

    resultados_kruskal = []
    for metrica in metricas_textura_cols:
        muestras = [texture_df.loc[texture_df["Grupo_Display"] == g, metrica].dropna().values for g in grupos_orden]
        muestras = [m for m in muestras if len(m) > 0]
        if len(muestras) < 2:
            continue
        estadistico, p_valor = kruskal(*muestras)
        resultados_kruskal.append({"Metrica": metrica, "H_Kruskal": estadistico, "p_valor": p_valor, "Significativo_p<0.05": "Si" if p_valor < 0.05 else "No"})

    kruskal_df = pd.DataFrame(resultados_kruskal)
    print(kruskal_df.to_string(index=False))

    print("\n" + "=" * 78)
    print("     COMPARACIONES POR PARES (Mann-Whitney U + correccion de Holm)")
    print("=" * 78)

    resultados_pares = []
    for metrica in metricas_textura_cols:
        for g1, g2 in combinations(grupos_orden, 2):
            d1 = texture_df.loc[texture_df["Grupo_Display"] == g1, metrica].dropna().values
            d2 = texture_df.loc[texture_df["Grupo_Display"] == g2, metrica].dropna().values
            if len(d1) == 0 or len(d2) == 0:
                continue
            U, p = mannwhitneyu(d1, d2, alternative="two-sided")
            n1, n2 = len(d1), len(d2)
            media_U = n1 * n2 / 2
            desviacion_U = np.sqrt(n1 * n2 * (n1 + n2 + 1) / 12)
            Z = (U - media_U) / desviacion_U if desviacion_U > 0 else np.nan
            r = abs(Z) / np.sqrt(n1 + n2) if not np.isnan(Z) else np.nan
            resultados_pares.append({"Metrica": metrica, "Comparacion": f"{g1} vs {g2}", "U": U, "p_valor": p, "Tamano_efecto_r": r})

    pares_df = pd.DataFrame(resultados_pares)
    if not pares_df.empty:
        _, p_ajustados, _, _ = multipletests(pares_df["p_valor"], method="holm")
        pares_df["p_valor_holm"] = p_ajustados
        pares_df["Significativo_p<0.05"] = pares_df["p_valor_holm"] < 0.05
        print(pares_df.round(4).to_string(index=False))
else:
    print("texture_df esta vacio, no hay pruebas estadisticas que correr todavia.")


## 13. Notas y siguientes pasos

- Si `WBCATT_IMAGES_DIR` no apunta a las imagenes reales del dataset PBC, todo lo que depende de abrir pixeles (secciones 4, 8-12) se quedara vacio para WBCAtt, pero las secciones 5 y 6 (distribucion de clases y atributos) funcionan igual porque solo dependen del CSV.
- Si `N_ATTR_LEUKEMIA` no coincide con lo que realmente traen tus `txt_labels`, el bloque de la seccion 7 te muestra el conteo real de columnas por linea (`conteo_columnas`) para que ajustes el numero.
- Los `json_labels` de LeukemiaAttri no se usaron aqui (se opto por `txt_labels` por ser mas simple de parsear), pero si necesitas los bounding boxes en formato COCO o metadata adicional, cada fila de `leukemia_df` trae `Carpeta_Condicion_Path`, y el json deberia estar en `Carpeta_Condicion_Path / "json_labels" / f"{stem}.json"` (o tambien como zip, en cuyo caso aplica la misma logica que `leer_lineas_txt_label`).
